# Arquitectura AI-Ready: Feature Stores y Streaming para ML

**Objetivo**: Diseñar e implementar arquitecturas de datos completas que soporten el ciclo de vida completo de ML, desde ingesta en tiempo real hasta inferencia en producción.

---

## 1. Introducción: Arquitectura por capas para ML

Una arquitectura AI-ready moderna se organiza en **capas especializadas**, cada una con responsabilidades claras:

```
┌────────────────────────────────────────────────────────────────────────┐
│                         CAPA DE INGESTA                                 │
│  ┌──────────────┐         ┌──────────────┐        ┌──────────────┐    │
│  │ Batch ETL    │         │ Streaming    │        │ Event Bus    │    │
│  │ (Airflow)    │         │ (Kafka/Flink)│        │ (Event Hub)  │    │
│  └──────────────┘         └──────────────┘        └──────────────┘    │
└──────────────────────────────┬─────────────────────────────────────────┘
                               ▼
┌────────────────────────────────────────────────────────────────────────┐
│                      CAPA DE ALMACENAMIENTO RAW                         │
│  ┌─────────────────────────────────────────────────────────────────┐  │
│  │ Data Lake (S3/ADLS)                                             │  │
│  │  • Bronze: Raw data (structured + unstructured)                 │  │
│  │  • Formato: Parquet, Avro, JSON, images, videos                │  │
│  └─────────────────────────────────────────────────────────────────┘  │
└──────────────────────────────┬─────────────────────────────────────────┘
                               ▼
┌────────────────────────────────────────────────────────────────────────┐
│                  CAPA DE FEATURE ENGINEERING                            │
│  ┌──────────────┐         ┌──────────────┐        ┌──────────────┐    │
│  │ Batch Jobs   │         │ Stream Proc. │        │ Transformers │    │
│  │ (Spark/Dask) │         │ (Flink/Beam) │        │ (dbt/Python) │    │
│  └──────────────┘         └──────────────┘        └──────────────┘    │
└──────────────────────────────┬─────────────────────────────────────────┘
                               ▼
┌────────────────────────────────────────────────────────────────────────┐
│                        FEATURE STORE                                    │
│  ┌──────────────────────────┐       ┌──────────────────────────┐      │
│  │ OFFLINE (Training)       │       │ ONLINE (Inference)       │      │
│  │ • Hive/Delta Lake        │       │ • Redis/DynamoDB         │      │
│  │ • High throughput        │       │ • Low latency (<10ms)    │      │
│  │ • Historical features    │       │ • Real-time features     │      │
│  └──────────────────────────┘       └──────────────────────────┘      │
└──────────────┬───────────────────────────────────┬─────────────────────┘
               ▼                                   ▼
   ┌─────────────────────┐            ┌─────────────────────┐
   │ TRAINING PIPELINE   │            │ INFERENCE SERVICE   │
   │ • Experiment track  │            │ • REST API          │
   │ • Model training    │            │ • Real-time predict │
   │ • Model registry    │            │ • A/B testing       │
   └─────────────────────┘            └─────────────────────┘
```

## 2. Feature Store: Arquitectura detallada

### 2.1 Dos capas: Offline vs Online

| Característica | Offline (Training) | Online (Inference) |
|----------------|--------------------|--------------------- |
| **Propósito** | Entrenamiento y experimentación | Predicciones en producción |
| **Latencia** | Minutos-horas | <10ms típicamente |
| **Volumen** | TB-PB (histórico completo) | KB-MB (últimos valores) |
| **Tecnología** | Hive, Delta Lake, Iceberg | Redis, DynamoDB, Aerospike |
| **Formato** | Parquet, ORC (columnar) | Key-Value, JSON |
| **Actualización** | Batch diario/semanal | Streaming (segundos) |
| **Consulta** | Spark SQL, Presto | Get by key |

### 2.2 Implementación de Feature Store

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Any, Optional
from datetime import datetime, timedelta
import random
import json

@dataclass
class FeatureDefinition:
    """Define una feature con sus metadatos"""
    name: str
    dtype: str  # int, float, string, categorical
    description: str
    source_table: str
    transformation: str
    update_frequency: str  # batch_daily, streaming, on_demand
    version: str

@dataclass
class FeatureValue:
    """Valor de una feature en un momento dado"""
    feature_name: str
    value: Any
    timestamp: datetime
    entity_id: str

class FeatureStoreAdvanced:
    """Feature Store con soporte offline y online"""
    
    def __init__(self):
        # Catálogo de features
        self.feature_catalog: Dict[str, FeatureDefinition] = {}
        
        # Almacenamiento offline (simulado como dict)
        # En producción: Hive/Delta Lake con particiones por fecha
        self.offline_store: Dict[str, List[FeatureValue]] = {}
        
        # Almacenamiento online (simulado como dict)
        # En producción: Redis con TTL
        self.online_store: Dict[str, FeatureValue] = {}
        
        # Registro de linaje
        self.lineage: List[Dict] = []
    
    def register_feature(self, feature_def: FeatureDefinition):
        """Registra una nueva feature en el catálogo"""
        self.feature_catalog[feature_def.name] = feature_def
        print(f"✓ Feature registrada: {feature_def.name} (v{feature_def.version})")
        print(f"  Fuente: {feature_def.source_table}")
        print(f"  Actualización: {feature_def.update_frequency}")
    
    def write_offline_batch(self, entity_id: str, features: Dict[str, Any], 
                            timestamp: datetime):
        """Escribe features en almacenamiento offline (batch)"""
        for feature_name, value in features.items():
            if feature_name not in self.feature_catalog:
                print(f"⚠️  Feature {feature_name} no está registrada")
                continue
            
            feature_value = FeatureValue(
                feature_name=feature_name,
                value=value,
                timestamp=timestamp,
                entity_id=entity_id
            )
            
            key = f"{entity_id}:{feature_name}"
            if key not in self.offline_store:
                self.offline_store[key] = []
            self.offline_store[key].append(feature_value)
        
        print(f"✓ Features escritas en OFFLINE para entity {entity_id} ({len(features)} features)")
    
    def write_online_streaming(self, entity_id: str, features: Dict[str, Any]):
        """Escribe features en almacenamiento online (streaming)"""
        timestamp = datetime.now()
        
        for feature_name, value in features.items():
            if feature_name not in self.feature_catalog:
                continue
            
            feature_value = FeatureValue(
                feature_name=feature_name,
                value=value,
                timestamp=timestamp,
                entity_id=entity_id
            )
            
            key = f"{entity_id}:{feature_name}"
            self.online_store[key] = feature_value
        
        print(f"✓ Features escritas en ONLINE para entity {entity_id} (latencia: 5ms)")
    
    def get_offline_features(self, entity_id: str, feature_names: List[str],
                             start_date: datetime, end_date: datetime) -> Dict[str, List[Any]]:
        """Recupera features históricas para entrenamiento"""
        result = {fname: [] for fname in feature_names}
        
        for feature_name in feature_names:
            key = f"{entity_id}:{feature_name}"
            if key in self.offline_store:
                # Filtrar por rango de fechas
                values = [
                    fv.value for fv in self.offline_store[key]
                    if start_date <= fv.timestamp <= end_date
                ]
                result[feature_name] = values
        
        print(f"→ Features OFFLINE recuperadas para training (entity {entity_id})")
        return result
    
    def get_online_features(self, entity_id: str, feature_names: List[str]) -> Dict[str, Any]:
        """Recupera features actuales para inferencia (baja latencia)"""
        result = {}
        
        for feature_name in feature_names:
            key = f"{entity_id}:{feature_name}"
            if key in self.online_store:
                result[feature_name] = self.online_store[key].value
            else:
                result[feature_name] = None
        
        print(f"→ Features ONLINE recuperadas para inference (latencia: 3ms)")
        return result
    
    def get_feature_lineage(self, feature_name: str) -> Dict:
        """Obtiene el linaje de una feature"""
        if feature_name not in self.feature_catalog:
            return {}
        
        feature_def = self.feature_catalog[feature_name]
        return {
            'feature': feature_name,
            'source': feature_def.source_table,
            'transformation': feature_def.transformation,
            'version': feature_def.version,
            'consumers': self._get_consumers(feature_name)
        }
    
    def _get_consumers(self, feature_name: str) -> List[str]:
        """Simula obtener qué modelos usan esta feature"""
        # En producción: consultar Model Registry
        return ["fraud_detection_v2", "churn_prediction_v1"]

# Demostración
print("="*70)
print("DEMO: Feature Store Offline + Online")
print("="*70)

fs = FeatureStoreAdvanced()

# Paso 1: Registrar features en el catálogo
print("\n📋 Paso 1: Registro de Features en Catálogo\n")

fs.register_feature(FeatureDefinition(
    name="user_avg_purchase_value_30d",
    dtype="float",
    description="Promedio de valor de compra en últimos 30 días",
    source_table="transactions",
    transformation="AVG(amount) OVER (PARTITION BY user_id ORDER BY timestamp ROWS 30 PRECEDING)",
    update_frequency="batch_daily",
    version="1.0"
))

fs.register_feature(FeatureDefinition(
    name="user_purchase_count_7d",
    dtype="int",
    description="Número de compras en últimos 7 días",
    source_table="transactions",
    transformation="COUNT(*) OVER (PARTITION BY user_id ORDER BY timestamp ROWS 7 PRECEDING)",
    update_frequency="streaming",
    version="1.0"
))

# Paso 2: Escribir features offline (batch para entrenamiento)
print("\n💾 Paso 2: Escritura Batch en OFFLINE Store\n")

for day in range(30):
    timestamp = datetime.now() - timedelta(days=30-day)
    fs.write_offline_batch(
        entity_id="user_12345",
        features={
            "user_avg_purchase_value_30d": random.uniform(50, 200),
            "user_purchase_count_7d": random.randint(1, 10)
        },
        timestamp=timestamp
    )

# Paso 3: Escribir features online (streaming para inferencia)
print("\n⚡ Paso 3: Escritura Streaming en ONLINE Store\n")

fs.write_online_streaming(
    entity_id="user_12345",
    features={
        "user_avg_purchase_value_30d": 125.50,
        "user_purchase_count_7d": 5
    }
)

# Paso 4: Recuperar features para entrenamiento
print("\n🎓 Paso 4: Recuperar Features para TRAINING\n")

training_features = fs.get_offline_features(
    entity_id="user_12345",
    feature_names=["user_avg_purchase_value_30d", "user_purchase_count_7d"],
    start_date=datetime.now() - timedelta(days=30),
    end_date=datetime.now()
)

print(f"  • user_avg_purchase_value_30d: {len(training_features['user_avg_purchase_value_30d'])} valores históricos")

# Paso 5: Recuperar features para inferencia
print("\n🚀 Paso 5: Recuperar Features para INFERENCE\n")

inference_features = fs.get_online_features(
    entity_id="user_12345",
    feature_names=["user_avg_purchase_value_30d", "user_purchase_count_7d"]
)

print("\nFeatures listas para predicción:")
for fname, value in inference_features.items():
    print(f"  • {fname}: {value}")

# Paso 6: Linaje
print("\n🔍 Paso 6: Linaje de Features\n")

lineage = fs.get_feature_lineage("user_avg_purchase_value_30d")
print(json.dumps(lineage, indent=2))

### 2.3 Comparativa de Feature Stores (2025)

| Feature Store | Offline | Online | Streaming | Cloud | Deployment | Mejor Para |
|---------------|---------|--------|-----------|-------|------------|------------|
| **Feast** | ✅ BigQuery, Redshift, Snowflake | ✅ Redis, DynamoDB, Datastore | ✅ Kafka, Kinesis | ☁️ Agnóstico | Self-managed | Open source, flexibilidad |
| **Tecton** | ✅ Databricks, Snowflake, Redshift | ✅ DynamoDB, Redis | ✅ Kafka, Kinesis, Flink | ☁️ Multi-cloud SaaS | Managed | Empresas, soporte completo |
| **AWS SageMaker FS** | ✅ S3 (offline store) | ✅ DynamoDB, RDS | ⚠️ Limitado | ☁️ Solo AWS | Managed (AWS) | Lock-in AWS aceptable |
| **Azure ML FS** | ✅ ADLS, SQL | ✅ Redis, Cosmos DB | ⚠️ Limitado | ☁️ Solo Azure | Managed (Azure) | Ecosistema Azure |
| **Vertex AI FS** | ✅ BigQuery | ✅ Bigtable | ✅ Pub/Sub | ☁️ Solo GCP | Managed (GCP) | Integración GCP |
| **Hopsworks** | ✅ Hive, Delta | ✅ RonDB | ✅ Kafka | ☁️ Hybrid | Self/Managed | On-prem + cloud |

#### Criterios de Selección

**Elige Feast si:**
- Necesitas flexibilidad y control total
- Tienes equipo para operarlo
- Presupuesto limitado (open source)
- Multi-cloud sin vendor lock-in

**Elige Tecton si:**
- Necesitas soporte empresarial 24/7
- Feature engineering avanzado (transformaciones stream)
- Observabilidad y monitoreo completo
- Presupuesto para SaaS (~$50K-500K/año)

**Elige Cloud Native (SageMaker/Azure/Vertex) si:**
- Ya estás comprometido con un cloud
- Quieres managed service sin operación
- Integración nativa con ML platform del proveedor
- Equipo pequeño sin expertise en infra

#### Ejemplo de Costos (Orden de Magnitud)

**Caso:** 1000 features, 100M requests/día

| Feature Store | Setup | Operación Mensual | Total Anual |
|---------------|-------|-------------------|-------------|
| Feast (self-managed) | $5K-10K (dev time) | $2K-5K (infra AWS) | ~$30K-70K |
| Tecton (SaaS) | $0 (onboarding incluido) | $10K-40K | ~$120K-480K |
| SageMaker FS | $0 (built-in) | $3K-8K (storage + queries) | ~$40K-100K |

**Trade-off:** Costos iniciales vs operacionales vs expertise requerido.

## 3. Integración de datos multimodales

### 3.1 Desafíos de datos multimodales

Combinar diferentes tipos de datos en un mismo pipeline de ML presenta desafíos específicos:

| Desafío | Ejemplo | Solución |
|---------|---------|----------|
| **Diferentes tasas de actualización** | Texto (tiempo real) vs Imágenes (batch) | Feature Store con múltiples fuentes |
| **Diferentes formatos** | CSV + JPEG + MP3 | Data Lake con zonas por tipo |
| **Diferentes procesadores** | NLP models + CNN + tradicional ML | Pipeline modular con orquestación |
| **Sincronización temporal** | Tweet + imagen del tweet | Event-driven con correlación por ID |

### 3.2 Arquitectura para datos multimodales

In [ ]:
from dataclasses import dataclass
from typing import Dict, Any, Optional
from datetime import datetime
import base64

@dataclass
class MultimodalData:
    """Representa un dato multimodal (ej: producto e-commerce)"""
    entity_id: str
    timestamp: datetime
    
    # Datos estructurados
    structured_data: Dict[str, Any]  # precio, categoría, stock, etc.
    
    # Datos no estructurados
    text_data: Optional[str] = None  # descripción, reseñas
    image_path: Optional[str] = None  # path a imagen en S3
    video_path: Optional[str] = None  # path a video en S3

class MultimodalPipeline:
    """Pipeline para procesar datos multimodales"""
    
    def __init__(self, feature_store: FeatureStoreAdvanced):
        self.feature_store = feature_store
        self.embeddings_cache = {}  # Cache de embeddings procesados
    
    def process_structured(self, data: Dict[str, Any]) -> Dict[str, float]:
        """Procesa features estructuradas"""
        features = {}
        
        # Features numéricas directas
        features['price'] = data.get('price', 0.0)
        features['stock'] = data.get('stock', 0)
        features['rating'] = data.get('rating', 0.0)
        
        # Features derivadas
        features['is_expensive'] = 1.0 if data.get('price', 0) > 100 else 0.0
        features['is_in_stock'] = 1.0 if data.get('stock', 0) > 0 else 0.0
        
        return features
    
    def process_text(self, text: str) -> Dict[str, float]:
        """Procesa texto y genera embeddings"""
        # En producción: usar BERT, GPT, etc.
        # Aquí simulamos embeddings
        features = {}
        
        # Simulación de embeddings (en producción: vector de 768 dimensiones)
        features['text_length'] = len(text)
        features['word_count'] = len(text.split())
        features['has_positive_words'] = 1.0 if any(word in text.lower() for word in ['excelente', 'bueno', 'recomendado']) else 0.0
        
        # Simular embedding vectorizado
        # features['text_embedding'] = [0.1, 0.2, ...] (768 dims)
        
        return features
    
    def process_image(self, image_path: str) -> Dict[str, float]:
        """Procesa imagen y extrae features visuales"""
        # En producción: usar CNN (ResNet, EfficientNet, etc.)
        features = {}
        
        # Simulación de features de imagen
        features['image_quality_score'] = 0.85  # Detectado por CNN
        features['has_people'] = 1.0  # Detección de objetos
        features['dominant_color_red'] = 0.6  # Análisis de color
        features['image_complexity'] = 0.7  # Entropía de la imagen
        
        # Simular embedding visual
        # features['image_embedding'] = [0.3, 0.1, ...] (2048 dims de ResNet)
        
        return features
    
    def process_multimodal(self, data: MultimodalData) -> Dict[str, float]:
        """Procesa todos los modos y combina features"""
        all_features = {}
        
        print(f"\n🔄 Procesando datos multimodales para: {data.entity_id}")
        
        # 1. Procesar estructurados
        if data.structured_data:
            struct_features = self.process_structured(data.structured_data)
            all_features.update({f"struct_{k}": v for k, v in struct_features.items()})
            print(f"  ✓ Features estructuradas: {len(struct_features)}")
        
        # 2. Procesar texto
        if data.text_data:
            text_features = self.process_text(data.text_data)
            all_features.update({f"text_{k}": v for k, v in text_features.items()})
            print(f"  ✓ Features de texto: {len(text_features)}")
        
        # 3. Procesar imagen
        if data.image_path:
            image_features = self.process_image(data.image_path)
            all_features.update({f"image_{k}": v for k, v in image_features.items()})
            print(f"  ✓ Features de imagen: {len(image_features)}")
        
        print(f"  → Total features combinadas: {len(all_features)}")
        
        return all_features
    
    def ingest_and_store(self, data: MultimodalData):
        """Ingesta datos multimodales y los almacena en Feature Store"""
        # Procesar todos los modos
        features = self.process_multimodal(data)
        
        # Guardar en Feature Store (offline para entrenamiento)
        # Nota: Solo guardamos features numéricas, no los embeddings completos aquí
        feature_subset = {k: v for k, v in features.items() if not k.endswith('_embedding')}
        
        self.feature_store.write_offline_batch(
            entity_id=data.entity_id,
            features=feature_subset,
            timestamp=data.timestamp
        )
        
        # Guardar en online para inferencia
        self.feature_store.write_online_streaming(
            entity_id=data.entity_id,
            features=feature_subset
        )

# Ejemplo de uso
print("="*70)
print("DEMO: Pipeline de Datos Multimodales")
print("="*70)

# Registrar features multimodales en el Feature Store
fs_multi = FeatureStoreAdvanced()

for feature_name in ['struct_price', 'struct_is_expensive', 'text_word_count', 
                      'text_has_positive_words', 'image_quality_score', 'image_has_people']:
    fs_multi.register_feature(FeatureDefinition(
        name=feature_name,
        dtype="float",
        description=f"Feature multimodal: {feature_name}",
        source_table="products_multimodal",
        transformation="various",
        update_frequency="streaming",
        version="1.0"
    ))

pipeline = MultimodalPipeline(fs_multi)

# Datos de un producto con múltiples modalidades
product_data = MultimodalData(
    entity_id="product_98765",
    timestamp=datetime.now(),
    structured_data={
        'price': 149.99,
        'stock': 25,
        'rating': 4.5,
        'category': 'electronics'
    },
    text_data="Auriculares inalámbricos excelentes con cancelación de ruido. Muy recomendados para trabajo remoto.",
    image_path="s3://products/images/product_98765.jpg",
    video_path="s3://products/videos/product_98765_demo.mp4"
)

# Procesar e ingestar
pipeline.ingest_and_store(product_data)

print("\n✅ Datos multimodales procesados y almacenados en Feature Store")

## 4. Caso práctico: Sistema de recomendación en streaming

Implementaremos un sistema completo de recomendación que:
1. Ingiere eventos de usuario en tiempo real (Kafka)
2. Calcula features en streaming (Flink/Spark Streaming)
3. Almacena en Feature Store (online/offline)
4. Sirve recomendaciones con baja latencia

### 4.1 Arquitectura del sistema

```
┌─────────────┐
│   Usuario   │
│   Clicks    │
│   Compras   │
└──────┬──────┘
       │ eventos en tiempo real
       ▼
┌─────────────────┐
│  Kafka Topic    │  ← user_events
│  (Event Stream) │
└────────┬────────┘
         │
         ▼
┌────────────────────────────────┐
│  Stream Processing             │
│  (Flink / Spark Streaming)     │
│                                │
│  • Ventanas temporales (5min)  │
│  • Agregaciones (count, avg)   │
│  • Join con catálogo productos │
└────────┬───────────────────────┘
         │
    ┌────┴─────┐
    ▼          ▼
┌────────┐  ┌──────────┐
│Offline │  │ Online   │
│Feature │  │ Feature  │
│Store   │  │ Store    │
│(Hive)  │  │ (Redis)  │
└───┬────┘  └────┬─────┘
    │            │
    ▼            ▼
┌────────┐  ┌──────────────┐
│Training│  │ Inference    │
│        │  │ API          │
│Model   │  │ GET /recos   │
└────────┘  └──────────────┘
```

In [ ]:
from dataclasses import dataclass
from typing import List, Dict, Any
from datetime import datetime, timedelta
from collections import defaultdict, deque
import random

@dataclass
class UserEvent:
    """Evento de usuario en tiempo real"""
    user_id: str
    event_type: str  # view, click, add_to_cart, purchase
    product_id: str
    timestamp: datetime
    metadata: Dict[str, Any]

class StreamProcessor:
    """Simula procesamiento de eventos en streaming (Flink/Spark Streaming)"""
    
    def __init__(self, window_size_minutes: int = 5):
        self.window_size = timedelta(minutes=window_size_minutes)
        
        # Ventanas temporales por usuario
        self.user_windows: Dict[str, deque] = defaultdict(lambda: deque(maxlen=100))
        
        # Features calculadas
        self.user_features: Dict[str, Dict] = {}
    
    def process_event(self, event: UserEvent) -> Dict[str, Any]:
        """Procesa un evento y actualiza features en tiempo real"""
        user_id = event.user_id
        
        # Agregar evento a la ventana
        self.user_windows[user_id].append(event)
        
        # Calcular features en ventana temporal
        features = self._calculate_window_features(user_id, event.timestamp)
        
        # Actualizar cache de features
        self.user_features[user_id] = features
        
        return features
    
    def _calculate_window_features(self, user_id: str, current_time: datetime) -> Dict[str, Any]:
        """Calcula features sobre ventana temporal"""
        window_start = current_time - self.window_size
        
        # Filtrar eventos en la ventana
        events_in_window = [
            e for e in self.user_windows[user_id]
            if e.timestamp >= window_start
        ]
        
        if not events_in_window:
            return self._default_features()
        
        # Calcular agregaciones
        features = {
            'events_count_5min': len(events_in_window),
            'views_count_5min': sum(1 for e in events_in_window if e.event_type == 'view'),
            'clicks_count_5min': sum(1 for e in events_in_window if e.event_type == 'click'),
            'add_to_cart_count_5min': sum(1 for e in events_in_window if e.event_type == 'add_to_cart'),
            'purchases_count_5min': sum(1 for e in events_in_window if e.event_type == 'purchase'),
        }
        
        # Features derivadas
        features['click_through_rate'] = (
            features['clicks_count_5min'] / features['views_count_5min']
            if features['views_count_5min'] > 0 else 0.0
        )
        
        features['conversion_rate'] = (
            features['purchases_count_5min'] / features['events_count_5min']
            if features['events_count_5min'] > 0 else 0.0
        )
        
        # Productos únicos vistos
        features['unique_products_5min'] = len(set(e.product_id for e in events_in_window))
        
        # Categorías más vistas (requiere join con catálogo de productos)
        # En producción: join con tabla de productos
        features['top_category'] = 'electronics'  # Simulado
        
        return features
    
    def _default_features(self) -> Dict[str, Any]:
        """Features por defecto cuando no hay eventos"""
        return {
            'events_count_5min': 0,
            'views_count_5min': 0,
            'clicks_count_5min': 0,
            'add_to_cart_count_5min': 0,
            'purchases_count_5min': 0,
            'click_through_rate': 0.0,
            'conversion_rate': 0.0,
            'unique_products_5min': 0,
            'top_category': 'unknown'
        }

class RecommendationSystem:
    """Sistema de recomendación completo con streaming"""
    
    def __init__(self):
        self.stream_processor = StreamProcessor(window_size_minutes=5)
        self.feature_store = FeatureStoreAdvanced()
        self.model_version = "v2.1"
        
        # Registrar features de streaming
        self._register_streaming_features()
    
    def _register_streaming_features(self):
        """Registra features de streaming en el catálogo"""
        streaming_features = [
            'events_count_5min', 'views_count_5min', 'clicks_count_5min',
            'click_through_rate', 'conversion_rate', 'unique_products_5min'
        ]
        
        for fname in streaming_features:
            self.feature_store.register_feature(FeatureDefinition(
                name=fname,
                dtype="float",
                description=f"Streaming feature: {fname}",
                source_table="user_events_stream",
                transformation="window aggregation",
                update_frequency="streaming",
                version="1.0"
            ))
    
    def ingest_event(self, event: UserEvent):
        """Ingesta un evento en tiempo real"""
        # 1. Procesar con stream processor
        features = self.stream_processor.process_event(event)
        
        # 2. Escribir en Feature Store online (para inferencia inmediata)
        self.feature_store.write_online_streaming(
            entity_id=event.user_id,
            features=features
        )
        
        # 3. También escribir en offline (para reentrenamiento)
        self.feature_store.write_offline_batch(
            entity_id=event.user_id,
            features=features,
            timestamp=event.timestamp
        )
    
    def get_recommendations(self, user_id: str, num_recommendations: int = 5) -> List[str]:
        """Obtiene recomendaciones para un usuario (inferencia)"""
        # 1. Obtener features del Feature Store online
        features = self.feature_store.get_online_features(
            entity_id=user_id,
            feature_names=['events_count_5min', 'click_through_rate', 'conversion_rate', 'unique_products_5min']
        )
        
        # 2. Inferencia del modelo (simulado)
        # En producción: llamar a modelo entrenado
        recommendations = self._model_predict(user_id, features, num_recommendations)
        
        return recommendations
    
    def _model_predict(self, user_id: str, features: Dict, num_recs: int) -> List[str]:
        """Simula predicción del modelo"""
        # En producción: usar modelo entrenado (collaborative filtering, deep learning, etc.)
        
        # Simulación: ajustar productos según actividad del usuario
        if features.get('click_through_rate', 0) > 0.3:
            # Usuario muy activo -> productos premium
            product_pool = ['prod_premium_001', 'prod_premium_002', 'prod_premium_003', 
                           'prod_premium_004', 'prod_premium_005']
        else:
            # Usuario menos activo -> productos populares
            product_pool = ['prod_popular_001', 'prod_popular_002', 'prod_popular_003',
                           'prod_popular_004', 'prod_popular_005']
        
        return random.sample(product_pool, min(num_recs, len(product_pool)))

# Demostración completa
print("="*70)
print("DEMO: Sistema de Recomendación en Streaming")
print("="*70)

rec_system = RecommendationSystem()

print("\n📊 Simulando eventos de usuario en tiempo real...\n")

# Simular secuencia de eventos
events = [
    UserEvent('user_123', 'view', 'prod_001', datetime.now(), {'page': 'home'}),
    UserEvent('user_123', 'click', 'prod_001', datetime.now(), {'source': 'recommendation'}),
    UserEvent('user_123', 'view', 'prod_002', datetime.now(), {'page': 'product'}),
    UserEvent('user_123', 'add_to_cart', 'prod_001', datetime.now(), {}),
    UserEvent('user_123', 'view', 'prod_003', datetime.now(), {'page': 'product'}),
    UserEvent('user_123', 'click', 'prod_003', datetime.now(), {'source': 'search'}),
]

for i, event in enumerate(events, 1):
    print(f"Evento {i}: {event.event_type} → {event.product_id}")
    rec_system.ingest_event(event)

print("\n🎯 Obteniendo recomendaciones para el usuario...\n")

recommendations = rec_system.get_recommendations('user_123', num_recommendations=5)

print("\n✨ Recomendaciones personalizadas:")
for i, prod_id in enumerate(recommendations, 1):
    print(f"  {i}. {prod_id}")

print("\n✅ Sistema de recomendación en streaming funcionando")
print("   • Latencia total: <50ms")
print("   • Features actualizadas en tiempo real")
print(f"   • Modelo: {rec_system.model_version}")

## 5. Arquitectura event-driven para ML

### 5.1 Patrones de arquitectura

Las arquitecturas event-driven son ideales para ML porque:

1. **Desacoplamiento**: Productores y consumidores independientes
2. **Escalabilidad**: Procesamiento paralelo de eventos
3. **Tiempo real**: Reacción inmediata a cambios
4. **Auditabilidad**: Event log completo para reproducibilidad

### 5.2 Componentes clave

```
Event Producers          Event Broker              Event Consumers
─────────────           ──────────────            ─────────────────

┌──────────┐            ┌─────────────┐           ┌──────────────┐
│   App    │───event───▶│   Kafka     │──────────▶│  Feature     │
│   Web    │            │   Topic 1   │           │  Engineering │
└──────────┘            └─────────────┘           └──────────────┘
                                │
┌──────────┐                    │                 ┌──────────────┐
│   IoT    │───event───▶        ├────────────────▶│  Model       │
│ Devices  │                    │                 │  Training    │
└──────────┘                    │                 └──────────────┘
                                │
┌──────────┐                    │                 ┌──────────────┐
│  Mobile  │───event───▶        └────────────────▶│  Real-time   │
│   App    │                                      │  Inference   │
└──────────┘                                      └──────────────┘
```

### 5.3 Ventajas para ML

| Ventaja | Beneficio para ML |
|---------|-------------------|
| **Event log inmutable** | Reproducibilidad de experimentos |
| **Replay de eventos** | Reentrenamiento con datos históricos |
| **Low latency** | Inferencia en tiempo real |
| **Schema evolution** | Cambios en features sin downtime |
| **Backpressure handling** | No perder eventos bajo alta carga |

## 6. Stack tecnológico recomendado

### Para implementar arquitectura AI-ready completa:

#### Ingesta y Streaming
- **Kafka/Confluent**: Event streaming
- **Airflow**: Orquestación de pipelines batch
- **Fluentd/Logstash**: Ingesta de logs

#### Procesamiento
- **Apache Flink**: Stream processing (estado, ventanas temporales)
- **Spark Streaming**: Micro-batch processing
- **dbt**: Transformaciones SQL

#### Feature Store
- **Tecton / Feast**: Feature stores open-source
- **Redis**: Online store (low latency)
- **Delta Lake/Iceberg**: Offline store (ACID)

#### ML Platform
- **MLflow**: Experiment tracking y model registry
- **Kubeflow**: Orquestación de workflows de ML
- **SageMaker/Vertex AI**: Plataformas cloud all-in-one

#### Inferencia
- **FastAPI**: APIs de inferencia
- **TorchServe/TF Serving**: Model serving especializado
- **Kubernetes**: Orquestación y escalado

#### Monitoreo
- **Prometheus + Grafana**: Métricas de sistema
- **Evidently AI**: Data drift y model drift
- **Great Expectations**: Validación de calidad de datos

## 7. Checklist de diseño

Al diseñar una arquitectura AI-ready, verifica:

### ✅ Datos
- [ ] Data Lake para almacenar raw data (estructurados + no estructurados)
- [ ] Soporte para ingesta batch y streaming
- [ ] Versionado de datasets
- [ ] Particionado eficiente (por fecha, por tipo)

### ✅ Feature Engineering
- [ ] Feature Store con capas offline y online
- [ ] Catálogo de features con linaje
- [ ] Reutilización de features entre proyectos
- [ ] Sincronización entre offline y online

### ✅ Entrenamiento
- [ ] Pipelines automatizados de entrenamiento
- [ ] Model registry con versionado
- [ ] Experiment tracking (métricas, hiperparámetros)
- [ ] Reproducibilidad (data + code + model version)

### ✅ Inferencia
- [ ] API de inferencia con SLA definido (<100ms típico)
- [ ] Features online con baja latencia
- [ ] Escalabilidad horizontal
- [ ] A/B testing de modelos

### ✅ Monitoreo
- [ ] Data drift detection
- [ ] Model performance monitoring
- [ ] Alertas automáticas
- [ ] Feedback loop (predicciones → datos)

### ✅ Gobernanza
- [ ] Linaje completo de datos
- [ ] Control de acceso granular
- [ ] Compliance con regulaciones (GDPR, etc.)
- [ ] Explainability de modelos

## 8. Casos de Uso Reales

### 8.1 Uber: Detección de Fraude en Tiempo Real

**Contexto:**
- 23M viajes/día
- Detectar fraude en <1 segundo
- Features: ubicación, historial conductor/pasajero, patrones de comportamiento

**Arquitectura:**
```
Eventos de viaje → Kafka → Flink (features streaming) → Michelangelo FS (online)
                                                              ↓
                                                    Modelo XGBoost (inferencia)
                                                              ↓
                                                    Bloqueo automático si score > 0.9
```

**Stack:**
- Feature Store: Michelangelo (propio de Uber, similar a Tecton)
- Streaming: Kafka + Flink
- Offline: Hive + Parquet
- Online: Cassandra (<10ms p99)

**Resultado:**
- Reducción 30% fraude
- Latencia p99: 8ms
- 50K+ features compartidas entre equipos

### 8.2 Netflix: Sistema de Recomendaciones

**Contexto:**
- 230M usuarios
- Personalización de contenido en homepage
- Features: historial visualización, ratings, contexto (hora, dispositivo)

**Arquitectura:**
```
┌─────────────┐     ┌──────────────┐     ┌─────────────────┐
│ User Events │────▶│ Kafka Stream │────▶│ Feature Store   │
│ (clicks)    │     │              │     │ Offline: S3     │
└─────────────┘     └──────────────┘     │ Online: EVCache │
                                         └────────┬────────┘
                                                  │
                                                  ▼
                                         ┌─────────────────┐
                                         │ Model Ensemble  │
                                         │ (5-10 modelos)  │
                                         └────────┬────────┘
                                                  │
                                                  ▼
                                         ┌─────────────────┐
                                         │ Personalized UI │
                                         └─────────────────┘
```

**Features clave:**
- Históricas: watch history, ratings (offline store)
- Real-time: sesión actual, hora del día (online store)
- Contextuales: dispositivo, ubicación geográfica

**Stack:**
- Streaming: Kafka
- Procesamiento: Flink + Spark
- Online Store: EVCache (memcached distribuido)
- Offline Store: S3 + Parquet
- ML Platform: Metaflow + TensorFlow

**Métricas:**
- 75% del contenido visto viene de recomendaciones
- Latencia p95 homepage: <200ms
- Re-training: cada 24h (batch) + updates incrementales (stream)

### 8.3 Airbnb: Precios Dinámicos (Dynamic Pricing)

**Contexto:**
- 7M listings en 220 países
- Sugerir precio óptimo por noche
- Features: ubicación, amenidades, demanda, eventos, temporada

**Arquitectura:**
```
Historical bookings → Airflow DAG → Spark Feature Engineering
                                              ↓
                                    ┌─────────────────────┐
                                    │  Zipline (FS)       │
                                    │  Offline: Hive      │
                                    │  Online: Redis      │
                                    └──────────┬──────────┘
                                               │
┌─────────────────┐                            ▼
│ Host requests   │                   ┌──────────────────┐
│ price suggest   │──────────────────▶│ Pricing Model    │
└─────────────────┘                   │ (LightGBM)       │
                                      └──────────┬───────┘
                                                 │
                                                 ▼
                                      ┌──────────────────┐
                                      │ Precio sugerido  │
                                      │ + confidence     │
                                      └──────────────────┘
```

**Features (300+):**
- **Listing:** #habitaciones, #baños, ubicación, amenities
- **Demand:** búsquedas recientes, bookings en área, eventos próximos
- **Host:** experiencia, ratings, cancellation rate
- **Temporal:** día semana, estacionalidad, holidays
- **Competencia:** precios similares en zona (scraped)

**Stack:**
- Feature Store: Zipline (interno Airbnb)
- Offline: Hive (PB-scale)
- Online: Redis (cientos de TBs)
- Training: Spark MLlib
- Serving: FastAPI + Kubernetes

**ROI:**
- +3% en bookings por mejor pricing
- Reducción 40% en tiempo de host para definir precio
- A/B testing continuo para optimizar modelo

### 8.4 Capital One: Aprobación de Crédito en Tiempo Real

**Contexto:**
- Millones de solicitudes de crédito/año
- Decisión en <5 segundos
- Features: credit score, historial pagos, ingresos, deuda actual

**Arquitectura:**
```
Application Form → API Gateway → Feature Store Online (DynamoDB)
                                            │
                                            ▼
                                  ┌──────────────────┐
                                  │ Credit Model     │
                                  │ (Logistic Reg +  │
                                  │  XGBoost)        │
                                  └────────┬─────────┘
                                           │
                                           ▼
                                  Approve / Deny / Manual Review
```

**Cumplimiento regulatorio:**
- Explainability obligatoria (Fair Credit Reporting Act)
- Linaje completo de features para auditoría
- Retención de decisiones 7 años
- Detección de sesgo (fairness metrics por demographic)

**Stack:**
- Feature Store: SageMaker Feature Store
- Online: DynamoDB + Redis caching
- ML Platform: SageMaker
- Explainability: SHAP values

**Métricas:**
- Latencia p99: 3.5 segundos
- 15% reducción en default rate vs modelo anterior
- 100% auditable por reguladores

### 8.5 Spotify: Discover Weekly (Playlist Personalizada)

**Contexto:**
- 500M usuarios
- Playlist semanal personalizada (30 canciones)
- Features: listening history, skips, likes, audio features

**Desafío único: Cold start**
- Usuarios nuevos sin historial → usar features contextuales
- Canciones nuevas sin plays → usar audio features (tempo, energy)

**Pipeline:**
```
User Listening Events → Kafka → Flink (aggregations)
                                      ↓
                            ┌──────────────────┐
                            │ Feature Store    │
                            │ User embeddings  │
                            │ Track embeddings │
                            └────────┬─────────┘
                                     │
                                     ▼
                            ┌──────────────────┐
                            │ Collaborative    │
                            │ Filtering +      │
                            │ Content-based    │
                            └────────┬─────────┘
                                     │
                                     ▼
                            Playlist Generation (Lunes AM)
```

**Features:**
- **User:** géneros favoritos, época del día, devices
- **Track:** audio features (Spotify API), popularidad, artista similar
- **Contexto:** trending tracks en país/región

**Stack:**
- Streaming: Kafka + Storm
- Feature Store: Custom (Redis + S3)
- Training: Spark + TensorFlow
- Embeddings: Word2Vec adaptado (Track2Vec)

**Resultado:**
- 40% de usuarios escuchan Discover Weekly cada semana
- 5B streams generados por recomendaciones/año

## 9. Observabilidad y Monitoreo de ML en Producción

### 9.1 Capas de Observabilidad

```
┌─────────────────────────────────────────────────────────────┐
│                  BUSINESS METRICS                            │
│  • Conversion rate      • Revenue impact                     │
│  • User engagement      • ROI del modelo                     │
└─────────────────────────────────────────────────────────────┘
                           ↓ ↑
┌─────────────────────────────────────────────────────────────┐
│                   MODEL METRICS                              │
│  • Accuracy / Precision / Recall                             │
│  • AUC-ROC              • F1-Score                          │
│  • Prediction distribution                                   │
└─────────────────────────────────────────────────────────────┘
                           ↓ ↑
┌─────────────────────────────────────────────────────────────┐
│                 DATA QUALITY METRICS                         │
│  • Feature freshness    • Null rates                         │
│  • Data drift           • Schema changes                     │
│  • Anomalies            • Completeness                       │
└─────────────────────────────────────────────────────────────┘
                           ↓ ↑
┌─────────────────────────────────────────────────────────────┐
│               OPERATIONAL METRICS                            │
│  • Latency (p50/p95/p99)  • Throughput (RPS)               │
│  • Error rate              • CPU/Memory usage               │
│  • Cache hit rate          • Queue depth                     │
└─────────────────────────────────────────────────────────────┘
```

### 9.2 Data Drift Detection

**Problema:** Distribución de features en producción diverge del training

**Solución:** Monitoreo continuo con tests estadísticos

| Método | Tipo de Data | Interpretación |
|--------|--------------|----------------|
| **Kolmogorov-Smirnov Test** | Numérica | p < 0.05 → drift detectado |
| **Chi-Square Test** | Categórica | p < 0.05 → drift detectado |
| **Population Stability Index (PSI)** | Numérica/Categórica | PSI > 0.25 → drift significativo |
| **Jensen-Shannon Divergence** | Ambas | JS > 0.2 → distribuciones divergentes |

**Ejemplo Conceptual:**
```python
from scipy.stats import ks_2samp

# Features en training
feature_train = [...]

# Features en producción (última semana)
feature_prod = [...]

# KS test
statistic, p_value = ks_2samp(feature_train, feature_prod)

if p_value < 0.05:
    alert("Data drift detectado en feature X")
    trigger_retraining_pipeline()
```

### 9.3 Model Performance Degradation

**Síntomas:**
- Accuracy baja gradualmente
- Distribución de predicciones cambia
- Aumento en false positives/negatives

**Estrategias:**
1. **Shadow Mode:** Nuevo modelo en paralelo sin servir predicciones
2. **Champion/Challenger:** Comparación A/B continua
3. **Canary Deployment:** Despliegue gradual (1% → 10% → 50% → 100%)

**Dashboard de ML Ops (concepto):**
```
┌────────────────────────────────────────────────────────────┐
│  Model: fraud_detection_v2.3  🟢 HEALTHY                   │
├────────────────────────────────────────────────────────────┤
│                                                            │
│  📊 Performance (Last 7d)                                  │
│     Precision:  94.2% ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓░░  [Target: 90%]  │
│     Recall:     89.5% ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓░░░  [Target: 85%]  │
│     F1-Score:   91.8% ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓░  [Target: 87%]  │
│                                                            │
│  🔍 Data Quality                                           │
│     Drift Score:  0.12 🟢  [Threshold: 0.25]              │
│     Null Rate:    0.3% ✓   [Target: <1%]                  │
│     Freshness:    5 min ✓  [SLA: <10 min]                 │
│                                                            │
│  ⚡ Operations                                             │
│     Latency p95:  45ms ✓   [SLA: <100ms]                  │
│     Throughput:   2.3K RPS                                │
│     Error Rate:   0.02% ✓  [Target: <0.1%]                │
│                                                            │
│  💰 Business Impact                                        │
│     Fraud detected:  1,234 cases                          │
│     $ Saved:         $2.5M (estimated)                    │
│     False pos rate:  2.1%  [Target: <5%]                  │
└────────────────────────────────────────────────────────────┘
```

### 9.4 Herramientas de Observabilidad ML

| Herramienta | Enfoque | Características |
|-------------|---------|-----------------|
| **Evidently AI** | Data + Model drift | Open source, drift reports automáticos |
| **Whylabs** | Data quality monitoring | SaaS, anomaly detection, whylogs SDK |
| **Arize AI** | ML observability completa | Model tracing, feature importance drift |
| **Fiddler AI** | Explainability + monitoring | Root cause analysis, fairness |
| **Grafana + Prometheus** | Operational metrics | Latencia, throughput, infra |

**Stack Recomendado (Open Source):**
```
Data Logging → Evidently → Prometheus → Grafana → Alertmanager → Slack/PagerDuty
```

## 10. Análisis de Costos y FinOps para ML

### 10.1 Desglose de Costos Típicos

Para un sistema ML en producción (escala media: 1M predicciones/día):

| Componente | % del Costo Total | Rango Mensual | Drivers de Costo |
|------------|-------------------|---------------|------------------|
| **Compute (Training)** | 30-40% | $5K-20K | GPUs, frecuencia retraining |
| **Compute (Inference)** | 15-25% | $3K-8K | Instancias, autoscaling |
| **Storage (Data Lake)** | 10-15% | $2K-5K | Volumen datos, retención |
| **Feature Store** | 15-20% | $3K-7K | Online (Redis/DynamoDB) costoso |
| **Streaming Platform** | 5-10% | $1K-3K | Kafka/Kinesis throughput |
| **Monitoreo/Logging** | 5-10% | $1K-3K | Volumen de logs, retention |
| **Data Transfer** | 5-10% | $1K-3K | Egress entre servicios/regiones |

**Total: $16K - $49K/mes** (~$200K - $600K/año)

### 10.2 Estrategias de Optimización

#### Training
```
❌ COSTOSO: GPU on-demand 24/7
  • p3.8xlarge (4x V100) = $12/hora × 24h × 30d = $8,640/mes

✅ OPTIMIZADO: Spot instances + scheduled training
  • p3.8xlarge spot (70% descuento) = $3.6/hora × 8h/día × 30d = $864/mes
  • Ahorro: $7,776/mes (90%)
```

#### Inference
```
❌ COSTOSO: Over-provisioned instances
  • 10x c5.2xlarge 24/7 = $0.34/h × 10 × 24 × 30 = $2,448/mes

✅ OPTIMIZADO: Autoscaling + serverless
  • 3x c5.2xlarge (baseline) + autoscaling to 8 (peak)
  • SageMaker Serverless (pay per invocation)
  • Ahorro: 40-60% vs over-provisioning
```

#### Feature Store Online
```
❌ COSTOSO: Redis/DynamoDB on-demand sin TTL
  • DynamoDB: $0.25/GB storage + $1.25/M reads = alto costo acumulativo

✅ OPTIMIZADO: TTL + caching inteligente + right-sizing
  • TTL 7 días (purga features antiguas)
  • Caching en API layer (80% cache hit rate)
  • Ahorro: 50-70%
```

#### Data Transfer
```
❌ COSTOSO: Cross-region replication continua
  • Egress $0.09/GB × 10TB/día = $27K/mes

✅ OPTIMIZADO: Regional compute + data locality
  • Procesar cerca de donde están los datos
  • Comprimir antes de transferir
  • Ahorro: 60-80%
```

### 10.3 ROI de Feature Store

**Caso:** Empresa con 5 equipos ML, 20 data scientists

#### Sin Feature Store
```
Desarrollo features:  5 equipos × 30% tiempo × $150K salary = $225K/año
Duplicación trabajo:  +40% overhead = $90K/año
Inconsistencias:      Debugging y remediación = $50K/año
────────────────────────────────────────────────────────────
Total costo "oculto": $365K/año
```

#### Con Feature Store
```
Costo Feature Store: $120K/año (Tecton SaaS o Feast self-managed)
Ahorro tiempo dev:   -50% tiempo features = $112K/año ahorrado
Reutilización:       80% features compartidas = $180K/año ahorrado
Reducción errores:   -70% inconsistencias = $35K/año ahorrado
────────────────────────────────────────────────────────────
Ahorro neto: $207K/año
ROI: 172%
Payback: 7 meses
```

### 10.4 Calculadora de Dimensionamiento

**Inputs típicos:**
- Predicciones/día: 1M
- Features por predicción: 50
- Latencia requerida: <100ms p95
- Training frequency: Diario

**Outputs estimados:**
```python
# Pseudocódigo de calculadora

def estimate_infrastructure(predictions_per_day, num_features, latency_sla_ms):
    # Inference compute
    rps = predictions_per_day / 86400  # requests per second
    instances = rps / 100  # assume 100 RPS per instance
    inference_cost = instances * 0.34 * 24 * 30  # c5.2xlarge pricing
    
    # Feature store online
    feature_size_kb = num_features * 0.1  # 100 bytes per feature
    total_storage_gb = (predictions_per_day * feature_size_kb) / (1024 * 1024)
    storage_cost = total_storage_gb * 0.25  # DynamoDB storage
    read_cost = predictions_per_day * 30 * (1.25 / 1_000_000)  # per M reads
    
    # Feature store offline
    historical_data_tb = (predictions_per_day * feature_size_kb * 365) / (1024**3)
    s3_cost = historical_data_tb * 23  # S3 standard per TB/month
    
    # Total
    total_monthly = inference_cost + storage_cost + read_cost + s3_cost
    return {
        'instances': instances,
        'inference_cost': inference_cost,
        'online_store_cost': storage_cost + read_cost,
        'offline_store_cost': s3_cost,
        'total_monthly': total_monthly,
        'total_annual': total_monthly * 12
    }

# Ejemplo
result = estimate_infrastructure(1_000_000, 50, 100)
print(f"Costo mensual estimado: ${result['total_monthly']:,.0f}")
print(f"Costo anual estimado: ${result['total_annual']:,.0f}")
```

### 10.5 Mejores Prácticas FinOps para ML

1. **Tagging granular:** Etiquetar recursos por equipo/proyecto/modelo
2. **Budget alerts:** Alertas al 80% del presupuesto mensual
3. **Right-sizing periódico:** Revisar utilización cada trimestre
4. **Spot instances para training:** Ahorro 60-90% en workloads tolerantes a interrupciones
5. **Lifecycle policies:** Mover datos cold a S3 Glacier (ahorro 80%)
6. **Federated compute:** Entrenar en múltiples clouds para arbitrar costos
7. **Cost allocation reports:** Chargeback por dominio/equipo para accountability

## Conclusión

### Puntos clave aprendidos

1. **Arquitectura por capas** es esencial:
   - Ingesta → Processing → Feature Store → Training/Inference
   - Cada capa con tecnologías especializadas

2. **Feature Store es el corazón**:
   - Dos capas (offline/online) para diferentes propósitos
   - Garantiza consistencia entre training e inference
   - Catálogo con linaje para gobernanza

3. **Datos multimodales** requieren:
   - Pipelines especializados por tipo de dato
   - Sincronización temporal
   - Embeddings para datos no estructurados

4. **Event-driven architecture** habilita:
   - Procesamiento en tiempo real
   - Desacoplamiento y escalabilidad
   - Reproducibilidad con event replay

5. **Sistema de recomendación completo** integra:
   - Streaming de eventos
   - Feature Store
   - Inferencia en tiempo real

### Próximos pasos

En el nivel Senior profundizaremos en:
- Gobernanza de modelos a escala empresarial
- Model registry y deployment strategies
- Taller de evaluación: sistema de mantenimiento predictivo completo
- Observabilidad de ML en producción
- Diseño de ML platform para múltiples equipos

---

**Referencias**:
- "Designing Data-Intensive Applications" - Martin Kleppmann
- "Feature Store for ML" - Jim Dowling & Tecton.ai
- "Streaming Systems" - Tyler Akidau et al.